# Checkpoint 4 — Contest 4 Practice Solutions

Worked answers for Questions 1–8, and a reference program for Question 9.
Each check cell works its answer out again with the tested helper from its unit: the graph tools from Unit 12, the circuit tools from Unit 13 (with the simplifier from Unit 8), and the assembly interpreter from Unit 15.
The two program questions are copied into the `assets` folder (`assets/trace5.py` and `assets/trace6.py`) and run.
The Question 9 program is saved as `assets/q9.py`; run it with `python assets/q9.py < input.txt`.
If you missed a question, go back to its unit: Questions 1–2 are Graph Theory, 3–4 are Digital Electronics, 5–6 are What Does This Program Do? – Strings, and 7–8 are Assembly Language.

## Question 1

List where each vertex can go next:

| From | To |
|---|---|
| A | B, E |
| B | C |
| C | A, D |
| D | A, E |
| E | B |

First find every cycle through `A`, by following the arrows out of `A` until they come back to it or get stuck.

- `A` → `B` → `C`, then `C` → `A` closes `ABCA`, and `C` → `D` → `A` closes `ABCDA`. (`D` → `E` → `B` is stuck: `B` is already used.)
- `A` → `E` → `B` → `C`, then `C` → `A` closes `AEBCA`, and `C` → `D` → `A` closes `AEBCDA`. (`D` → `E` is stuck: `E` is already used.)

Then the cycles that avoid `A`: without `A` the only edges are `BC`, `CD`, `DE` and `EB`, which make the one cycle `BCDEB`.

That is 5 cycles, each counted once: `ABCA`, `ABCDA`, `AEBCA`, `AEBCDA`, `BCDEB`.

**Answer:** `5`

In [ ]:
import sys

sys.path.insert(0, "../../units/unit-12-graph-theory/assets/verify")
import graph_eval

assert str(len(graph_eval.cycles("{AB, AE, BC, CA, CD, DA, DE, EB}", directed=True))) == "5"

## Question 2

A path of length 3 from `C` to `E` is `C` → `x` → `y` → `E`: read row `C` for `x`, row `x` for `y`, and then check that `y` has an edge to `E`.

| C → x | x → y | y → E? | path |
|---|---|---|---|
| A | B | yes | `CABE` |
| A | D | yes | `CADE` |
| D | B | yes | `CDBE` |
| D | E | no (row `E` has no 1 in column `E`) | |

So the entry in row `C` and column `E` of `M³` is 3.

**Answer:** `3`

In [ ]:
import sys

sys.path.insert(0, "../../units/unit-12-graph-theory/assets/verify")
import graph_eval

#      A  B  C  D  E
M = [[0, 1, 0, 1, 0],  # A
     [0, 0, 1, 0, 1],  # B
     [1, 0, 0, 1, 0],  # C
     [0, 1, 0, 0, 1],  # D
     [1, 0, 0, 0, 0]]  # E
assert str(graph_eval.matrix_power(M, 3)[2][4]) == "3"

## Question 3

The output is `r * B`, so `B` must be 1.
`r` is `NAND(p, q)`, which is 1 unless both `p` and `q` are 1.
With `B` = 1, `p` = `A ⊕ 1` is 1 exactly when `A` = 0, and `q` = `~C` is 1 exactly when `C` = 0.
So `r` is 0 only for `A` = 0 and `C` = 0.

| A B C | p | q | r | out |
|---|---|---|---|---|
| 0 1 0 | 1 | 1 | 0 | 0 |
| 0 1 1 | 1 | 0 | 1 | 1 |
| 1 1 0 | 0 | 1 | 1 | 1 |
| 1 1 1 | 0 | 0 | 1 | 1 |

Every row with `B` = 0 is FALSE because of the last `AND`.

**Answer:** `(0,1,1), (1,1,0), (1,1,1)`

In [ ]:
import sys

sys.path.insert(0, "../../units/unit-13-digital-electronics/assets/verify")
import circuit_eval

netlist = """INPUTS A B C
p = XOR(A, B)
q = NOT(C)
r = NAND(p, q)
out = AND(r, B)"""
assert str(circuit_eval.solutions(netlist)) == "(0,1,1), (1,1,0), (1,1,1)"

## Question 4

The expression is `(~(A + B) + A * C) ⊙ C`, where `⊙` is XNOR: it is 1 when `r` and `C` agree.

| A B C | ~(A + B) | A * C | r | out = r ⊙ C |
|---|---|---|---|---|
| 0 0 0 | 1 | 0 | 1 | 0 |
| 0 0 1 | 1 | 0 | 1 | 1 |
| 0 1 0 | 0 | 0 | 0 | 1 |
| 0 1 1 | 0 | 0 | 0 | 0 |
| 1 0 0 | 0 | 0 | 0 | 1 |
| 1 0 1 | 0 | 1 | 1 | 1 |
| 1 1 0 | 0 | 0 | 0 | 1 |
| 1 1 1 | 0 | 1 | 1 | 1 |

The circuit is FALSE only in rows `000` and `011`.

- `A` alone covers the four rows `100, 101, 110, 111`, all TRUE.
- Row `001` needs a term: `~A * ~B` would cover `000` and `~A * C` would cover `011`, so the only two-letter choice is `~B * C` (rows `001, 101`).
- Row `010` likewise: `~A * B` covers `011` and `~A * ~C` covers `000`, so it must be `B * ~C` (rows `010, 110`).
- Those two terms are forced, and they leave `100` and `111`, which one term can cover only as `A`.

Three terms and five letters, and no other choice is as short.
In order: `A`, then `B * ~C`, then `~B * C` (`B` before `~B`).

**Answer:** `A + B * ~C + ~B * C`

In [ ]:
import sys

sys.path.insert(0, "../../units/unit-13-digital-electronics/assets/verify")
sys.path.insert(0, "../../units/unit-08-boolean-algebra/assets/verify")
import bool_eval
import circuit_eval

netlist = """INPUTS A B C
p = NOR(A, B)
q = AND(A, C)
r = OR(p, q)
out = XNOR(r, C)"""
minimal = bool_eval.minimal_sops(circuit_eval.to_expression(netlist))
assert len(minimal) == 1
assert str(minimal[0]) == "A + B * ~C + ~B * C"

## Question 5

Number the letters: `W`0 `A`1 `T`2 `E`3 `R`4 `M`5 `E`6 `L`7 `O`8 `N`9.
`T` starts as `S[:1]`, the first letter: `W`.
In the loop, `S[j:j + 1]` is positions `j` through `j + 1` (two letters), and `S[j:]` is the **last** `j` letters.

| j | S[j:j + 1] | S[j:] | T |
|---|---|---|---|
| 1 | `AT` | `N` | `WATN` |
| 2 | `TE` | `ON` | `WATNTEON` |
| 3 | `ER` | `LON` | `WATNTEONERLON` |

`T` has 4 + 4 + 5 = 13 letters, and the output puts one space between `T` and its length.

**Answer:** `WATNTEONERLON 13`

The check cell runs a Python copy of the program (`assets/trace5.py`): `FOR j = 1 TO 3` becomes `range(1, 4)`, the ACSL `S[j:j + 1]` becomes the Python slice `S[j:j + 2]`, and the last `j` letters become `S[len(S) - j:]`.

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, "assets/trace5.py"], input='', capture_output=True, text=True)
output = result.stdout.rstrip("\n")
assert str(output) == 'WATNTEONERLON 13'

## Question 6

The `while` loop finds `k`, the position of `S[j]` in the alphabet (`A` is 0).
Then the letter `j` places further on is added, wrapping from `Z` back to `A`.

| j | S[j] | k | (k + j) % 26 | letter |
|---|---|---|---|---|
| 0 | `Z` | 25 | 25 | `Z` |
| 1 | `E` | 4 | 5 | `F` |
| 2 | `B` | 1 | 3 | `D` |
| 3 | `R` | 17 | 20 | `U` |
| 4 | `A` | 0 | 4 | `E` |

`T` is `ZFDUE`.
This is Python, so `T[1:4]` is positions 1, 2 and 3: `FDU`.

**Answer:** `ZFDUE FDU`

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, "assets/trace6.py"], input='', capture_output=True, text=True)
output = result.stdout.rstrip("\n")
assert str(output) == 'ZFDUE FDU'

## Question 7

| Instruction | ACC | Z |
|---|---|---|
| `LOAD X` | -23 | 0 |
| `DIV Y` | -23 ÷ 4 = -5.75, rounded toward zero: -5 | 0 |
| `STORE Z` | -5 | -5 |
| `MULT Y` | -20 | -5 |
| `SUB X` | -20 - (-23) = 3 | -5 |
| `MULT =-1` | -3 | -5 |
| `ADD Z` | -3 + (-5) = -8 | -5 |
| `STORE Z` | -8 | -8 |

The trap is `DIV`: it rounds toward zero, so -23 ÷ 4 is -5, not -6.
(The middle steps compute the remainder: -5 × 4 - (-23) = 3, and its negative is -3.)

**Answer:** `-8`

In [ ]:
import sys

sys.path.insert(0, "../../units/unit-15-assembly-language/assets/verify")
import asm_eval

program = """X     DC    -23
Y     DC    4
Z     DC    0
      LOAD  X
      DIV   Y
      STORE Z
      MULT  Y
      SUB   X
      MULT  =-1
      ADD   Z
      STORE Z
      END"""
assert str(asm_eval.run(program)["memory"]["Z"]) == "-8"

## Question 8

Each pass reads `N`.
A negative `N` ends the loop and prints `C`.
Otherwise `N - 10` is tested: above 0 prints `N`; 0 or less adds `N` to `C`.

| N | N - 10 | action | C |
|---|---|---|---|
| 7 | -3 | add | 7 |
| 15 | 5 | print 15 | 7 |
| 3 | -7 | add | 10 |
| 10 | 0 | add (0 is not greater than 0) | 20 |
| 22 | 12 | print 22 | 20 |
| -1 | | branch to `DONE`, print 20 | 20 |

**Answer:** `15 22 20`

In [ ]:
import sys

sys.path.insert(0, "../../units/unit-15-assembly-language/assets/verify")
import asm_eval

program = """C     DC    0
LOOP  READ  N
      LOAD  N
      BL    DONE
      SUB   =10
      BG    BIG
      LOAD  C
      ADD   N
      STORE C
      BU    LOOP
BIG   PRINT N
      BU    LOOP
DONE  PRINT C
      END"""
printed = asm_eval.run(program, [7, 15, 3, 10, 22, -1])["printed"]
assert str(" ".join(str(v) for v in printed)) == "15 22 20"

## Question 9

**Notice:** use the adjacency-matrix idiom from Unit 12, but for a **directed** graph: set only `m[a][b]` for the edge `AB`, never `m[b][a]`.
Find each letter's row by walking along the first line, so the letters need not start at `A`.
For each route, first check every step `m[x][y]`; if any step is 0, the route is `NOT A PATH`.
Otherwise count the repeated pairs, the pairs of positions `i < j` holding the same vertex: no repeats is a `SIMPLE PATH`, exactly one repeat that is the first and last letters is a `CYCLE`, and anything else is a `PATH`.
Test the edges before the repeats: `AA` repeats a letter, but with no edge from `A` to itself it is `NOT A PATH`.
The fixtures include the two-vertex cycle `ABA` beside the two-letter paths `AB` and `BA`, the route `ABAB` that goes round twice, a square where the cycle can start at any vertex (`PQRSP` and `RSPQR`), edges that only go one way (`AB CB`, so `BA` fails), vertices with no edges at all, a larger graph with 40 routes, and all 26 letters with 120 edges and 400 routes.

In [ ]:
names = input().split()[0]
edges = input().split()
n = len(names)
m = []
for i in range(n):
    row = []
    for j in range(n):
        row.append(0)
    m.append(row)
for e in edges:
    a = 0
    while names[a] != e[0]:
        a = a + 1
    b = 0
    while names[b] != e[1]:
        b = b + 1
    m[a][b] = 1
routes = int(input())
for r in range(routes):
    route = input().split()[0]
    places = []
    for letter in route:
        a = 0
        while names[a] != letter:
            a = a + 1
        places.append(a)
    joined = True
    for i in range(len(places) - 1):
        if m[places[i]][places[i + 1]] == 0:
            joined = False
    repeats = 0
    for i in range(len(places)):
        for j in range(i + 1, len(places)):
            if places[i] == places[j]:
                repeats = repeats + 1
    if not joined:
        print("NOT A PATH")
    elif repeats == 0:
        print("SIMPLE PATH")
    elif repeats == 1 and places[0] == places[len(places) - 1]:
        print("CYCLE")
    else:
        print("PATH")